In [2]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")

df = pd.DataFrame(dataset['train'])

print("Jumlah total data:", len(df))
print("Daftar nama kolom:", list(df.columns))

display(df.head())

kolom_label = df.columns[1] 
print("\nDistribusi Label Sentimen:")
print(df[kolom_label].value_counts())

Jumlah total data: 11606
Daftar nama kolom: ['review', 'translate', 'sentimen']


,review,translate,sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,sound and mic satisfied. I only ordered the on...,0
1,Good. Mudah-mudahan awet. Makasih,good. hopefully durable. Thanks,1
2,"kondisi dus robek2, tolong di perhatikan lagi","the condition of the box is torn, please pay a...",0
3,Kualitas produk baik. Harga terjangkau dan pen...,product quality is good. affordable prices and...,0
4,"Overall bagus, tombol berfungsi semuanya, tapi...","overall is good, the buttons all work, but the...",0



Distribusi Label Sentimen:
translate
delivery time                                                                                                                                                                                                                                                                               2
delivery is a bit long                                                                                                                                                                                                                                                                      2
nice                                                                                                                                                                                                                                                                                        2
the item is ok, but the delivery is quite long                                                          

In [3]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def preprocessing_teks(teks):
    teks = str(teks).lower()
    
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks)
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    
    tokens = teks.split()
    
    tokens = [kata for kata in tokens if kata not in stopwords]
    
    hasil_stem = [stemmer.stem(kata) for kata in tokens]
    
    return " ".join(hasil_stem)

print("Fungsi preprocessing berhasil dibuat!")

Fungsi preprocessing berhasil dibuat!


In [4]:
pd.set_option('display.max_colwidth', None)

kolom_teks = df.columns[0]

df['teks_sesudah_preprocessing'] = df[kolom_teks].apply(preprocessing_teks)

tabel_perbandingan = df[[kolom_teks, 'teks_sesudah_preprocessing']].head(10)
display(tabel_perbandingan)

,review,teks_sesudah_preprocessing
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg ga nyala
1,Good. Mudah-mudahan awet. Makasih,good mudah mudah awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek tolong perhati
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk harga jangkau kirim cepet banget udh order oke tp sayang klo kado gabisa nitip pake kartu ucap bungkus pake kertas kado
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol fungsi packaging penyot robek
5,"Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.",produk original kualitas cepat kirim sanagt cepat recommended seller
6,kualitas sangat jelek. bocor,kualitas jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka",terimakasih ka barang terima sukses ka
8,"Body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli CM 1659 penasaran Aja, yang cm 1579 udah mau rusak mesin nya, Harga lebih murah 4k dari cm 1579 Fast delivery, courier Ramah sangat, ty seller",body berat cm spiral tipis banding cm moga awet beli cm penasaran aja cm udah rusak mesin harga murah cm fast delivery courier ramah ty seller
9,pesanan di terima tanggal 27 January 2021 tetapi baru bisa saya buka pagi ini tanggal 29 January 2021 pukul 06.00 dan ternyata tab SKETSA ADVAN tidak bisa di nyalakan sama sekali dan di cas juga posisi Tab tidak bisa menyala,pesan terima tanggal january buka pagi tanggal january tab sketsa advan nyala cas posisi tab nyala


### Pengaruh Preprocessing terhadap Kualitas Data Teks
1. Mengurangi Noise: Tahap cleaning menghapus URL, angka, tanda baca, dan simbol yang tidak memiliki makna sentimen sehingga data menjadi lebih bersih.
2. Menyeragamkan Variasi Kata: Tahap case folding dan stemming menyatukan bentuk kata yang berbeda tetapi bermakna sama (contoh: "Membeli", "membeli", dan "dibeli" diubah menjadi kata dasar "beli"), sehingga mengurangi jumlah kosakata unik (vocabulary) yang berlebihan.
3. Meningkatkan Fokus Informasi: Tahap stopword removal membuang kata hubung atau kata umum berfrekuensi tinggi (seperti "yang", "dan", "di"), sehingga kata-kata kunci pembawa sentimen lebih menonjol untuk tahap pemodelan.